# DC vs Verification on AIME 2024

**Goal**: Decide whether to build a full verifier pipeline or commit to cheatsheet-only.

| # | Condition | Description | Budget |
|---|-----------|-------------|--------|
| 1 | Baseline | CoT Pass@1 | 30 |
| 2 | Self-Consistency | N=16 majority vote | 480 |
| 3 | Dynamic Cheatsheet (DC) | N=8, reflect+curate, self-consistency selects | ~480 |
| 4 | DC + Verification | Same, but only verified-correct strategies enter playbook | ~480 |

3 seeds x 4 conditions. Qwen2.5-7B-Instruct via vLLM on A100.

**GPU efficiency**: Model download overlaps pip install. Baseline+SC mega-batched across all seeds. DC/DC+V seeds run concurrently.

## 1. Install + Model Pre-download (overlapped)

In [ ]:
%%bash
# Phase 1: Install huggingface_hub first (fast, ~10s)
pip install -q huggingface_hub 2>/dev/null

# Phase 2: Background-download model weights while vllm installs
# Model is ~14GB; this runs concurrently with pip install vllm
huggingface-cli download Qwen/Qwen2.5-7B-Instruct --quiet &
MODEL_DL_PID=$!
echo "Model download started in background (PID: $MODEL_DL_PID)"

# Phase 3: Install vllm + deps (the slow part, ~3-5 min)
pip install vllm==0.6.6 openai==1.58.1 nest_asyncio==1.6.0 2>&1 | tail -5

# Wait for model download to finish
echo "Waiting for model download to complete..."
wait $MODEL_DL_PID 2>/dev/null
echo "Done. Model cached, vllm installed."

## 2. Start vLLM Server

In [ ]:
import subprocess, time, os, signal

VLLM_PORT = 8000
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

vllm_log = open('/tmp/vllm_server.log', 'w')
vllm_proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL_NAME,
        "--port", str(VLLM_PORT),
        "--dtype", "bfloat16",
        "--gpu-memory-utilization", "0.95",
        "--max-model-len", "8192",
        "--max-num-seqs", "1024",
        "--max-num-batched-tokens", "16384",
        "--enable-prefix-caching",
        "--disable-log-requests",
    ],
    stdout=vllm_log,
    stderr=subprocess.STDOUT,
)
print(f"vLLM starting (PID: {vllm_proc.pid}). Loading model + compiling CUDA graphs...")
print("Meanwhile, next cell defines all experiment code.")

## 3. Define Experiment Code (runs while vLLM loads)

This cell is pure Python — no GPU needed. Run it immediately while vLLM starts up.

In [ ]:
import asyncio
import copy
import csv
import json
import math
import os
import random
import re
import time
import nest_asyncio
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

nest_asyncio.apply()

# =========================================================================
# Answer Parsing (from ShinkaEvolve/examples/adas_aime/utils.py)
# =========================================================================
ANSWER_REGEX = re.compile(r"-?\d+(?:,\d{3})*(?:\.\d+)?")

def extract_numeric_answer(text: str) -> str:
    matches = ANSWER_REGEX.findall(text.replace(",", ""))
    return matches[-1].lstrip("0") if matches else text.strip()

def last_boxed_only_string(string: str) -> str:
    idx = string.rfind("\\boxed")
    if idx < 0:
        idx = string.rfind("\\fbox")
    if idx < 0:
        return ""
    brace_idx = string.find("{", idx)
    if brace_idx < 0:
        return ""
    level = 0
    for i in range(brace_idx, len(string)):
        if string[i] == "{":
            level += 1
        elif string[i] == "}":
            level -= 1
            if level == 0:
                return string[idx : i + 1]
    return ""

def clean_answer(s):
    s = s.replace("\\dfrac", "\\frac")
    s = s.replace("x \\in", "")
    s = re.sub(r"\\mathbf\s*{([^}]*)}", r"\1", s)
    s = re.sub(r"\\textbf\s*{([^}]*)}", r"\1", s)
    return s

def remove_boxed(s):
    if "\\boxed " in s:
        left = "\\boxed "
        assert s[: len(left)] == left
        return s[len(left) :]
    left = "\\boxed{"
    if not s.startswith(left):
        return None
    assert s[-1] == "}"
    return clean_answer(s[len(left) : -1])

def fix_fracs(string):
    substrs = string.split("\\frac")
    new_str = substrs[0]
    if len(substrs) > 1:
        for substr in substrs[1:]:
            new_str += "\\frac"
            if substr[0] == "{":
                new_str += substr
            else:
                try:
                    assert len(substr) >= 2
                except AssertionError:
                    return string
                a, b = substr[0], substr[1]
                if b != "{":
                    new_str += "{" + a + "}{" + b + "}" + substr[2:]
                else:
                    new_str += "{" + a + "}" + b + substr[2:]
    return new_str

def fix_a_slash_b(string):
    if len(string.split("/")) != 2:
        return string
    a, b = string.split("/")
    try:
        a, b = int(a), int(b)
        assert string == "{}/{}".format(a, b)
        return "\\frac{" + str(a) + "}{" + str(b) + "}"
    except (AssertionError, ValueError):
        return string

def fix_sqrt(string):
    if "\\sqrt" not in string:
        return string
    splits = string.split("\\sqrt")
    new_string = splits[0]
    for split in splits[1:]:
        if split[0] != "{":
            new_string += "\\sqrt{" + split[0] + "}" + split[1:]
        else:
            new_string += "\\sqrt" + split
    return new_string

def remove_right_units(string):
    if "\\text{ " in string:
        splits = string.split("\\text{ ")
        assert len(splits) == 2
        return splits[0]
    return string

def strip_string(string):
    string = string.replace("\n", "")
    string = string.replace("\\!", "")
    string = string.replace("\\\\", "\\")
    string = string.replace("tfrac", "frac")
    string = string.replace("dfrac", "frac")
    string = string.replace("\\left", "")
    string = string.replace("\\right", "")
    string = string.replace("^{\\circ}", "")
    string = string.replace("^\\circ", "")
    string = string.replace("\\$", "")
    string = remove_right_units(string)
    string = string.replace("\\%", "")
    string = string.replace("%", "")
    string = string.replace(" .", " 0.")
    string = string.replace("{.", "{0.")
    if len(string) == 0:
        return string
    if string[0] == ".":
        string = "0" + string
    if len(string.split("=")) == 2:
        if len(string.split("=")[0]) <= 2:
            string = string.split("=")[1]
    string = fix_sqrt(string)
    string = string.replace(" ", "")
    string = fix_fracs(string)
    if string == "0.5":
        string = "\\frac{1}{2}"
    if string == "5.5":
        string = "\\frac{11}{2}"
    string = fix_a_slash_b(string)
    return string

def is_equiv(str1, str2, verbose=False):
    if str1 is None and str2 is None:
        return True
    if str1 is None or str2 is None:
        return False
    try:
        ss1 = strip_string(str1)
        ss2 = strip_string(str2)
        return ss1 == ss2
    except Exception:
        return str1 == str2

def parse_answer(raw: str) -> str:
    """Extract answer: try \\boxed first, then #### pattern, then last number."""
    boxed = last_boxed_only_string(raw)
    if boxed:
        inner = remove_boxed(boxed)
        if inner is not None:
            return inner.strip()
    m = re.search(r"####\s*(-?[\d,]+\.?\d*)", raw)
    if m:
        return m.group(1).replace(",", "").strip()
    return extract_numeric_answer(raw)

def check_answer(predicted: str, ground_truth: str) -> bool:
    if is_equiv(predicted, ground_truth):
        return True
    try:
        return int(float(predicted.replace(",", ""))) == int(float(ground_truth.replace(",", "")))
    except (ValueError, TypeError):
        return False

# =========================================================================
# Playbook
# =========================================================================
@dataclass
class Bullet:
    id: str
    section: str
    content: str
    helpful: int = 0
    harmful: int = 0

    def to_str(self) -> str:
        return f"[{self.id}] helpful={self.helpful} harmful={self.harmful} :: {self.content}"

@dataclass
class Playbook:
    bullets: List[Bullet] = field(default_factory=list)
    _next_id: int = 1

    def add(self, section: str, content: str) -> str:
        prefix = {"STRATEGIES": "str", "COMMON_MISTAKES": "err", "SOLUTION_PATTERNS": "sol"}.get(section, "gen")
        bid = f"{prefix}-{self._next_id:05d}"
        self._next_id += 1
        self.bullets.append(Bullet(id=bid, section=section, content=content))
        return bid

    def remove(self, bid: str):
        self.bullets = [b for b in self.bullets if b.id != bid]

    def update(self, bid: str, content: str):
        for b in self.bullets:
            if b.id == bid:
                b.content = content
                return

    def tag(self, bid: str, label: str):
        for b in self.bullets:
            if b.id == bid:
                if label == "helpful":
                    b.helpful += 1
                elif label == "harmful":
                    b.harmful += 1

    def to_str(self) -> str:
        sections = defaultdict(list)
        for b in self.bullets:
            sections[b.section].append(b.to_str())
        parts = []
        for sec in ["STRATEGIES", "COMMON_MISTAKES", "SOLUTION_PATTERNS"]:
            if sections[sec]:
                parts.append(f"## {sec}")
                parts.extend(sections[sec])
        return "\n".join(parts) if parts else "(empty playbook)"

    def copy(self) -> "Playbook":
        return copy.deepcopy(self)

    @property
    def size(self) -> int:
        return len(self.bullets)

MAX_BULLETS = 20

def make_initial_playbook() -> Playbook:
    pb = Playbook()
    pb.add("STRATEGIES", "AIME problems have integer answers from 000 to 999. Always give a non-negative integer.")
    pb.add("STRATEGIES", "Break complex problems into smaller sub-problems and solve each step carefully.")
    pb.add("COMMON_MISTAKES", "Watch for off-by-one errors in counting and combinatorics problems.")
    return pb

# =========================================================================
# LLM Interface
# =========================================================================
MAX_CONCURRENT = 64
_semaphore = asyncio.Semaphore(MAX_CONCURRENT)
call_counter = defaultdict(int)

# Will be initialized after vLLM is ready
aclient = None

async def llm_call_async(system: str, user: str, role: str = "generate",
                         temperature: float = 0.7, max_tokens: int = 2048) -> str:
    call_counter[role] += 1
    async with _semaphore:
        try:
            resp = await aclient.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            print(f"LLM call failed ({role}): {e}")
            return ""

# =========================================================================
# Generate / Reflect / Curate
# =========================================================================
def _build_generate_system(playbook: Optional[Playbook] = None) -> str:
    base = (
        "You are an expert math competition solver. Solve the problem step-by-step.\n"
        "Show all your work clearly. At the end, put your final integer answer inside \\boxed{}.\n"
        "AIME answers are always integers from 0 to 999.\n"
    )
    if playbook and playbook.size > 0:
        base += f"\nPLAYBOOK (use these strategies, reference IDs like [str-00001]):\n{playbook.to_str()}"
    return base

async def generate_one(question: str, playbook: Optional[Playbook] = None,
                       temperature: float = 0.7) -> Tuple[str, List[str], str]:
    system = _build_generate_system(playbook)
    user = f"Solve this AIME problem:\n\n{question}"
    raw = await llm_call_async(system, user, role="generate", temperature=temperature)
    answer = parse_answer(raw)
    bullets_used = list(set(re.findall(r"\[(str|err|sol|gen)-\d{5}\]", raw)))
    return answer, bullets_used, raw

async def generate_n(question: str, n: int, playbook: Optional[Playbook] = None,
                     temperature: float = 0.7) -> List[Tuple[str, List[str], str]]:
    return await asyncio.gather(*[generate_one(question, playbook, temperature) for _ in range(n)])

def majority_vote(answers: List[str]) -> str:
    counter = Counter()
    for a in answers:
        try:
            counter[str(int(float(a.replace(",", ""))))] += 1
        except (ValueError, TypeError):
            counter[a.strip()] += 1
    return counter.most_common(1)[0][0] if counter else ""

async def reflect_async(question, raw_response, predicted, ground_truth,
                        bullets_used, playbook, is_correct):
    feedback = "CORRECT" if is_correct else f"INCORRECT (predicted {predicted}, expected {ground_truth})"
    bullets_text = "\n".join(f"  {b.to_str()}" for b in playbook.bullets if b.id in bullets_used)
    if not bullets_text:
        bullets_text = "  (none referenced)"
    system = (
        "You are a math reasoning analyst. Analyze the solution and whether playbook strategies helped.\n"
        'For each bullet ID used, output a JSON line: {"id": "str-00001", "tag": "helpful"}\n'
        "Tags: helpful, harmful, neutral.\n"
        "End with a reflection paragraph about what mathematical insight was key."
    )
    user = (
        f"Problem: {question}\n\n"
        f"Solution:\n{raw_response}\n\n"
        f"Result: {feedback}\n\n"
        f"Bullets referenced:\n{bullets_text}"
    )
    raw = await llm_call_async(system, user, role="reflect", temperature=0.3)
    tags = {}
    for m in re.finditer(r'"id"\s*:\s*"([^"]+)".*?"tag"\s*:\s*"(helpful|harmful|neutral)"', raw):
        bid, tag = m.group(1), m.group(2)
        if bid in bullets_used:
            tags[bid] = tag
    if not tags and bullets_used:
        default_tag = "helpful" if is_correct else "harmful"
        tags = {bid: default_tag for bid in bullets_used}
    return raw, tags

async def curate_async(playbook, reflection, question):
    pb = playbook.copy()
    pb_text = pb.to_str()
    system = (
        "You are a playbook curator for math competition solving. Based on the reflection, "
        "propose operations to improve the playbook.\n"
        "Output a JSON array of operations:\n"
        '[{"op": "ADD", "section": "STRATEGIES", "content": "new insight"},\n'
        ' {"op": "UPDATE", "id": "str-00001", "content": "refined text"},\n'
        ' {"op": "DELETE", "id": "err-00002"}]\n'
        f"Sections: STRATEGIES, COMMON_MISTAKES, SOLUTION_PATTERNS\n"
        f"Max bullets: {MAX_BULLETS}. Current: {pb.size}.\n"
        "Only propose operations clearly supported by the reflection. Keep it minimal."
    )
    user = f"Question: {question}\nCurrent playbook:\n{pb_text}\n\nReflection:\n{reflection}"
    raw = await llm_call_async(system, user, role="curator", temperature=0.7)
    json_match = re.search(r"\[.*\]", raw, re.DOTALL)
    ops = []
    if json_match:
        try:
            ops = json.loads(json_match.group())
        except json.JSONDecodeError:
            pass
    orig_next = pb._next_id
    for op in ops:
        try:
            if op.get("op") == "ADD" and pb.size < MAX_BULLETS:
                pb.add(op.get("section", "STRATEGIES"), op.get("content", ""))
            elif op.get("op") == "UPDATE" and op.get("id"):
                pb.update(op["id"], op.get("content", ""))
            elif op.get("op") == "DELETE" and op.get("id"):
                pb.remove(op["id"])
        except Exception:
            pass
    if pb.size == 0:
        pb = make_initial_playbook()
        pb._next_id = orig_next
    return pb

print("All experiment code defined.")
print(f"Initial playbook ({make_initial_playbook().size} bullets):")
print(make_initial_playbook().to_str())

## 4. Wait for vLLM + Load Data

In [ ]:
from openai import OpenAI, AsyncOpenAI

client = OpenAI(base_url=f"http://localhost:{VLLM_PORT}/v1", api_key="dummy")
aclient = AsyncOpenAI(base_url=f"http://localhost:{VLLM_PORT}/v1", api_key="dummy")

print("Waiting for vLLM server...")
for attempt in range(300):
    if vllm_proc.poll() is not None:
        with open('/tmp/vllm_server.log') as f:
            print(f"vLLM DIED (exit={vllm_proc.returncode}). Log tail:")
            print(f.read()[-2000:])
        raise RuntimeError("vLLM process died")
    try:
        client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": "Hi"}],
            max_tokens=1,
        )
        print(f"vLLM ready after ~{attempt*2}s")
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("vLLM failed to start in 10 min")

# Warmup: trigger CUDA graph compilation
print("Warming up (CUDA graph compilation)...")
warmup = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "What is 2+2? Answer in \\boxed{}."}],
    max_tokens=64,
)
print(f"Warmup: {warmup.choices[0].message.content.strip()[:80]}")
print(f"vLLM running (PID: {vllm_proc.pid})")

In [ ]:
# Load AIME 2024 — upload the CSV or fetch from repo
import io, urllib.request

# Try local path first, then download from repo
CSV_URL = "https://raw.githubusercontent.com/ShengranHu/ADAS/main/data/AIME_Dataset_1983_2025.csv"
LOCAL_CSV = "/content/AIME_Dataset_1983_2025.csv"

# Check multiple possible locations
csv_paths = [
    "AIME_Dataset_1983_2025.csv",
    "ShinkaEvolve/examples/adas_aime/AIME_Dataset_1983_2025.csv",
    LOCAL_CSV,
]

csv_content = None
for path in csv_paths:
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            csv_content = f.read()
        print(f"Loaded from {path}")
        break

if csv_content is None:
    print(f"Downloading from {CSV_URL}...")
    try:
        csv_content = urllib.request.urlopen(CSV_URL).read().decode("utf-8")
        with open(LOCAL_CSV, "w") as f:
            f.write(csv_content)
        print(f"Saved to {LOCAL_CSV}")
    except Exception as e:
        print(f"Download failed: {e}")
        print("Please upload AIME_Dataset_1983_2025.csv manually.")
        from google.colab import files
        uploaded = files.upload()
        fname = list(uploaded.keys())[0]
        csv_content = uploaded[fname].decode("utf-8")

# Parse AIME 2024
reader = csv.DictReader(io.StringIO(csv_content))
all_problems = []
for row in reader:
    if str(row["Year"]).strip() == "2024":
        all_problems.append({
            "id": row["ID"],
            "problem": row["problem"],
            "answer": str(int(row["answer"])),
        })

print(f"\nLoaded {len(all_problems)} AIME 2024 problems")
assert len(all_problems) == 30, f"Expected 30, got {len(all_problems)}"

# Pre-compute shuffled orderings for all seeds
SEEDS = [42, 123, 7]
shuffled_problems = {}
for seed in SEEDS:
    rng = random.Random(seed)
    sp = all_problems.copy()
    rng.shuffle(sp)
    shuffled_problems[seed] = sp
    print(f"Seed {seed}: {[p['id'] for p in sp[:5]]}...")

print("\nData ready. Proceeding to experiment.")

## 5. Phase 1: Baseline + Self-Consistency (mega-batch all seeds)

These conditions are stateless — no playbook evolution. Fire ALL calls across all 3 seeds simultaneously.
- Baseline: 3 seeds × 30 problems × 1 = **90 calls**
- Self-Consistency: 3 seeds × 30 problems × 16 = **1,440 calls**
- Total: **1,530 calls** in one async batch → maximum GPU saturation

In [ ]:
call_counter = defaultdict(int)
t0 = time.time()

# Build ALL tasks upfront
baseline_tasks = {}   # (seed, problem_idx) -> task
sc_tasks = {}         # (seed, problem_idx) -> [task * 16]

all_async_tasks = []
task_index = []  # track what each task corresponds to

for seed in SEEDS:
    for i, p in enumerate(shuffled_problems[seed]):
        # Baseline: 1 call, temperature=0
        all_async_tasks.append(generate_one(p["problem"], playbook=None, temperature=0.0))
        task_index.append(("baseline", seed, i))

        # Self-Consistency: 16 calls, temperature=0.7
        for _ in range(16):
            all_async_tasks.append(generate_one(p["problem"], playbook=None, temperature=0.7))
            task_index.append(("sc", seed, i))

print(f"Launching {len(all_async_tasks)} calls (90 baseline + 1440 SC)...")
all_responses = await asyncio.gather(*all_async_tasks)
elapsed = time.time() - t0
print(f"Phase 1 complete: {len(all_async_tasks)} calls in {elapsed:.0f}s ({len(all_async_tasks)/elapsed:.1f} calls/s)")

# Unpack results
baseline_results = {seed: [] for seed in SEEDS}
sc_raw = {seed: {i: [] for i in range(30)} for seed in SEEDS}

for (cond, seed, idx), resp in zip(task_index, all_responses):
    answer, bullets, raw = resp
    if cond == "baseline":
        p = shuffled_problems[seed][idx]
        correct = check_answer(answer, p["answer"])
        baseline_results[seed].append({
            "id": p["id"], "predicted": answer, "ground_truth": p["answer"],
            "correct": correct, "raw": raw,
        })
    elif cond == "sc":
        sc_raw[seed][idx].append(answer)

# Aggregate SC results
sc_results = {seed: [] for seed in SEEDS}
for seed in SEEDS:
    for i, p in enumerate(shuffled_problems[seed]):
        answers = sc_raw[seed][i]
        winner = majority_vote(answers)
        correct = check_answer(winner, p["answer"])
        sc_results[seed].append({
            "id": p["id"], "predicted": winner, "ground_truth": p["answer"],
            "correct": correct, "n_candidates": 16,
            "answer_distribution": dict(Counter(answers).most_common()),
        })

# Print Phase 1 summary
print("\n" + "="*60)
print("PHASE 1 RESULTS")
print("="*60)
for seed in SEEDS:
    b_acc = sum(1 for r in baseline_results[seed] if r["correct"]) / 30
    s_acc = sum(1 for r in sc_results[seed] if r["correct"]) / 30
    print(f"Seed {seed:3d}: Baseline={b_acc:.0%}  SC(N=16)={s_acc:.0%}")

phase1_calls = dict(call_counter)
phase1_time = elapsed
print(f"\nTotal calls: {sum(phase1_calls.values())}  Time: {elapsed:.0f}s")

## 6. Phase 2: Dynamic Cheatsheet (3 seeds concurrent)

DC is sequential per-problem (playbook evolves), but the 3 seeds are independent.
Run all 3 concurrently → 3×8=24 generate calls per step + reflect + curate.

In [ ]:
async def run_dc_single_seed(problems, seed, n_candidates=8):
    """Run DC for one seed. Sequential across problems, parallel within."""
    playbook = make_initial_playbook()
    results = []
    for i, p in enumerate(problems):
        responses = await generate_n(p["problem"], n_candidates, playbook=playbook, temperature=0.7)
        answers = [r[0] for r in responses]
        winner = majority_vote(answers)
        correct = check_answer(winner, p["answer"])

        # Find response matching majority vote for reflection
        best_raw, best_bullets = responses[0][2], responses[0][1]
        for ans, bullets, raw in responses:
            try:
                if str(int(float(ans.replace(",", "")))) == winner:
                    best_raw, best_bullets = raw, bullets
                    break
            except (ValueError, TypeError):
                if ans.strip() == winner:
                    best_raw, best_bullets = raw, bullets
                    break

        # Reflect using self-consistency verdict (NO ground truth)
        reflection, tags = await reflect_async(
            p["problem"], best_raw, winner, "N/A (self-consistency)",
            best_bullets, playbook, is_correct=True
        )
        for bid, tag in tags.items():
            playbook.tag(bid, tag)
        playbook = await curate_async(playbook, reflection, p["problem"])

        results.append({
            "id": p["id"], "predicted": winner, "ground_truth": p["answer"],
            "correct": correct, "problem_idx": i, "playbook_size": playbook.size,
            "n_candidates": n_candidates,
            "answer_distribution": dict(Counter(answers).most_common()),
        })
        print(f"  DC seed={seed} [{i+1}/30] {p['id']}: {'Y' if correct else 'N'} (pred={winner} gt={p['answer']} pb={playbook.size})")
    return results

call_counter = defaultdict(int)
t0 = time.time()
print("Running DC: 3 seeds concurrently...")

dc_all = await asyncio.gather(*[
    run_dc_single_seed(shuffled_problems[seed], seed)
    for seed in SEEDS
])
dc_results = {seed: res for seed, res in zip(SEEDS, dc_all)}

dc_time = time.time() - t0
dc_calls = dict(call_counter)
print(f"\nPhase 2 complete: {sum(dc_calls.values())} calls in {dc_time:.0f}s")
for seed in SEEDS:
    acc = sum(1 for r in dc_results[seed] if r["correct"]) / 30
    print(f"  Seed {seed}: DC accuracy = {acc:.0%}")

## 7. Phase 3: DC + Verification (3 seeds concurrent)

Same as DC, but only verified-correct strategies enter the playbook.
Ground truth = perfect verifier (ceiling estimate).

In [ ]:
async def run_dcv_single_seed(problems, seed, n_candidates=8):
    """Run DC+Verification for one seed. Uses ground truth as perfect verifier."""
    playbook = make_initial_playbook()
    results = []
    for i, p in enumerate(problems):
        responses = await generate_n(p["problem"], n_candidates, playbook=playbook, temperature=0.7)
        answers = [r[0] for r in responses]
        winner = majority_vote(answers)
        correct = check_answer(winner, p["answer"])

        # Find a CORRECT response (ground truth verification)
        verified_raw, verified_bullets, any_correct = "", [], False
        for ans, bullets, raw in responses:
            if check_answer(ans, p["answer"]):
                verified_raw, verified_bullets, any_correct = raw, bullets, True
                break

        if any_correct:
            reflection, tags = await reflect_async(
                p["problem"], verified_raw, p["answer"], p["answer"],
                verified_bullets, playbook, is_correct=True
            )
        else:
            reflection, tags = await reflect_async(
                p["problem"], responses[0][2], winner, p["answer"],
                responses[0][1], playbook, is_correct=False
            )

        for bid, tag in tags.items():
            playbook.tag(bid, tag)
        playbook = await curate_async(playbook, reflection, p["problem"])

        results.append({
            "id": p["id"], "predicted": winner, "ground_truth": p["answer"],
            "correct": correct, "any_candidate_correct": any_correct,
            "problem_idx": i, "playbook_size": playbook.size,
            "n_candidates": n_candidates,
            "answer_distribution": dict(Counter(answers).most_common()),
        })
        print(f"  DC+V seed={seed} [{i+1}/30] {p['id']}: {'Y' if correct else 'N'} (v={any_correct} pb={playbook.size})")
    return results

call_counter = defaultdict(int)
t0 = time.time()
print("Running DC+V: 3 seeds concurrently...")

dcv_all = await asyncio.gather(*[
    run_dcv_single_seed(shuffled_problems[seed], seed)
    for seed in SEEDS
])
dcv_results = {seed: res for seed, res in zip(SEEDS, dcv_all)}

dcv_time = time.time() - t0
dcv_calls = dict(call_counter)
print(f"\nPhase 3 complete: {sum(dcv_calls.values())} calls in {dcv_time:.0f}s")
for seed in SEEDS:
    acc = sum(1 for r in dcv_results[seed] if r["correct"]) / 30
    print(f"  Seed {seed}: DC+V accuracy = {acc:.0%}")

## 8. Analysis + Decision

In [ ]:
# Assemble all results into unified structure
all_results = []
for seed in SEEDS:
    r = {
        "seed": seed, "n_problems": 30,
        "baseline": {
            "results": baseline_results[seed],
            "accuracy": sum(1 for x in baseline_results[seed] if x["correct"]) / 30,
        },
        "self_consistency": {
            "results": sc_results[seed],
            "accuracy": sum(1 for x in sc_results[seed] if x["correct"]) / 30,
        },
        "dc": {
            "results": dc_results[seed],
            "accuracy": sum(1 for x in dc_results[seed] if x["correct"]) / 30,
        },
        "dc_verified": {
            "results": dcv_results[seed],
            "accuracy": sum(1 for x in dcv_results[seed] if x["correct"]) / 30,
        },
    }
    all_results.append(r)

# === Summary Table ===
print("=" * 70)
print("DC vs Verification on AIME 2024 — Results")
print("=" * 70)
print(f"Seeds: {SEEDS}  |  Problems: 30  |  Model: {MODEL_NAME}")
print()

conditions = ["baseline", "self_consistency", "dc", "dc_verified"]
labels = {
    "baseline": "Baseline (CoT Pass@1)",
    "self_consistency": "Self-Consistency (N=16)",
    "dc": "Dynamic Cheatsheet (DC)",
    "dc_verified": "DC + Outcome Verification",
}

stats = {}
print(f"{'Condition':35s} {'Mean':>7s} {'Std':>7s}  {'Per-seed':>20s}")
print("-" * 70)
for cond in conditions:
    accs = [r[cond]["accuracy"] for r in all_results]
    mean_acc = sum(accs) / len(accs)
    std_acc = (sum((a - mean_acc)**2 for a in accs) / len(accs)) ** 0.5
    stats[cond] = {"mean": mean_acc, "std": std_acc}
    seeds_str = "  ".join(f"{a:.0%}" for a in accs)
    print(f"{labels[cond]:35s} {mean_acc:6.1%} {std_acc:6.1%}   [{seeds_str}]")

# === Learning Curves ===
print("\n" + "-" * 70)
print("DC Learning Curve (first 10 vs last 10 problems):")
for cond in ["dc", "dc_verified"]:
    per_problem = []
    for i in range(30):
        c = sum(1 for r in all_results if r[cond]["results"][i]["correct"])
        per_problem.append(c / len(all_results))
    first10 = sum(per_problem[:10]) / 10
    last10 = sum(per_problem[-10:]) / 10
    print(f"  {labels[cond]:35s}: first10={first10:.0%}  last10={last10:.0%}  delta={last10-first10:+.0%}")

# === Oracle Analysis ===
print("\n" + "-" * 70)
print("DC+V Oracle (any of N=8 candidates correct):")
for r in all_results:
    oracle = sum(1 for x in r["dc_verified"]["results"] if x.get("any_candidate_correct", False)) / 30
    actual = r["dc_verified"]["accuracy"]
    print(f"  Seed {r['seed']}: oracle={oracle:.0%}  actual={actual:.0%}  (selection gap={oracle-actual:+.0%})")

# === DECISION ===
print("\n" + "=" * 70)
print("DECISION")
print("=" * 70)
dc_mean = stats["dc"]["mean"]
dcv_mean = stats["dc_verified"]["mean"]
sc_mean = stats["self_consistency"]["mean"]
gap = dcv_mean - dc_mean

print(f"  DC accuracy:           {dc_mean:.1%}")
print(f"  DC+V accuracy:         {dcv_mean:.1%}")
print(f"  Self-Consistency:      {sc_mean:.1%}")
print(f"  Gap (DC+V - DC):       {gap:+.1%}")
print()

if gap < 0.05:
    verdict, text = "A", "Verifier over-engineered"
    next_step = "Commit to DC-only. Focus on cheatsheet quality, strategy diversity."
elif gap < 0.15:
    verdict, text = "A+", "Marginal value"
    next_step = "DC-first with lightweight verification (self-consistency as proxy)."
else:
    verdict, text = "B", "Verifier justified"
    next_step = "Build distilled verifier pipeline. Invest in verifier diversity."

print(f"  VERDICT: Path {verdict} — {text}")
print(f"  NEXT:    {next_step}")
print()

dc_vs_sc = dc_mean - sc_mean
print(f"  DC vs SC gap: {dc_vs_sc:+.1%}")
if dc_vs_sc > 0.05:
    print("  -> Playbook learning adds value beyond just sampling more.")
elif dc_vs_sc > -0.05:
    print("  -> Playbook roughly matches extra sampling (compute-neutral).")
else:
    print("  -> Playbook hurts — investigate strategy quality.")

# === Timing ===
total_time = phase1_time + dc_time + dcv_time
print(f"\n  Timing: Phase1={phase1_time:.0f}s  DC={dc_time:.0f}s  DC+V={dcv_time:.0f}s  Total={total_time:.0f}s")

## 9. Save Results

In [ ]:
# Save full results JSON
results_json = json.dumps(all_results, indent=2, default=str)
with open("dc_vs_verification_results.json", "w") as f:
    f.write(results_json)
print(f"Saved dc_vs_verification_results.json ({len(results_json)} bytes)")

# Save analysis summary
analysis = {
    "stats": stats,
    "gap_dcv_minus_dc": gap,
    "verdict": verdict,
    "verdict_text": text,
    "next_step": next_step,
    "timing": {"phase1_s": phase1_time, "dc_s": dc_time, "dcv_s": dcv_time, "total_s": total_time},
}
with open("dc_vs_verification_analysis.json", "w") as f:
    json.dump(analysis, f, indent=2)
print("Saved dc_vs_verification_analysis.json")

# Download
try:
    from google.colab import files
    files.download("dc_vs_verification_results.json")
    files.download("dc_vs_verification_analysis.json")
except Exception:
    print("(Not in Colab — files saved locally)")

In [ ]:
# Cleanup: kill vLLM server
try:
    os.kill(vllm_proc.pid, signal.SIGTERM)
    print(f"Killed vLLM server (PID {vllm_proc.pid})")
except ProcessLookupError:
    print("vLLM server already stopped")